[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the_ai_engineer_capstones/blob/main/capstones/week01_gd_optimization/gd_capstone.ipynb)

# Gradient Descent on a Piecewise Non-Smooth Objective — Week 1 Capstone

> This notebook implements the Week-1 capstone for *The AI Engineer* program.
> We study gradient descent on the piecewise non-smooth objective
>
> $$f(x) = \left|\tfrac{1}{2}x^3 - \tfrac{3}{2}x^2\right| + \tfrac{1}{2}x$$
>
> which has a kink (non-differentiability) at $x = 3$ and a global minimizer at
> $x^\star = 1 - \tfrac{2\sqrt{3}}{3} \approx -0.155$.
>
> We also study the convex quadratic baseline $q(x) = \tfrac{1}{2}x^2$ as a
> clean reference for step-size stability.

**Sections:**
1. Setup & Hyperparameters
2. Objective Function & Derivative
3. Function Landscape
4. Gradient Check (finite-difference validation)
5. Gradient Descent Implementation
6. GD Trajectories — Multiple Initializations
7. Step-Size Sweep
8. Quadratic Baseline — Stability Reference
9. Final Commentary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 123
rng = np.random.default_rng(SEED)

X_STAR = 1 - (2/3)*np.sqrt(3)   # ≈ −0.1547, global minimizer
EPS    = 1e-6                     # finite-difference step for gradient check
TOL    = 1e-4                     # convergence tolerance
T_MAX  = 5000

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print(f'X_STAR = {X_STAR:.6f}')

## Hyperparameter Reference

| Symbol | Value | Role |
|--------|-------|------|
| `SEED` | `123` | Global RNG seed — ensures fully reproducible trajectories |
| `X_STAR` | $1 - \tfrac{2\sqrt{3}}{3} \approx -0.1547$ | Analytic global minimizer of $f$ |
| `EPS` | `1e-6` | Finite-difference step for gradient check |
| `TOL` | `1e-4` | Convergence criterion: $f(x_t) - f(x^\star) < \text{TOL}$ |
| `T_MAX` | `5000` | Maximum iteration count |
| $\eta$ (GD sweep) | `0.01, 0.05, 0.10, 0.20, 0.50` | Step sizes used for the piecewise $f$ sweep |
| $\eta$ (quadratic) | `0.05, 0.10, 0.15, 0.20, 2.1` | Step sizes used for the quadratic baseline |

## 2. Objective Function & Derivative

Define $g(x) = \tfrac{1}{2}x^3 - \tfrac{3}{2}x^2 = \tfrac{x^2(x-3)}{2}$. The objective is

$$f(x) = |g(x)| + \tfrac{1}{2}x.$$

**Why the kink at $x = 3$.**  
The absolute value creates a kink wherever $g(x) = 0$ *and* $g'(x) \neq 0$.
At $x = 3$: $g(3) = 0$ and $g'(3) = \tfrac{3}{2}(9) - 3(3) = 4.5 \neq 0$, so $x = 3$ is a genuine kink.
At $x = 0$: $g(0) = 0$ and $g'(0) = 0$, so $f$ is smooth at $x = 0$.

**Piecewise derivative** (valid off the kink):

$$f'(x) = \operatorname{sign}(g(x))\cdot g'(x) + \tfrac{1}{2}, \qquad g'(x) = \tfrac{3}{2}x^2 - 3x.$$

**Global minimizer.**  
On the branch $g(x) < 0$ (which includes $x \approx -0.155$): $f'(x) = -g'(x) + \tfrac{1}{2} = -\tfrac{3}{2}x^2 + 3x + \tfrac{1}{2}$.
Setting $f'(x) = 0$ and solving the quadratic gives $x^\star = 1 - \tfrac{2\sqrt{3}}{3} \approx -0.1547$.

In [ ]:
def g(x):
    return 0.5*x**3 - 1.5*x**2

def f(x):
    return np.abs(g(x)) + 0.5*x

def df(x):
    # Piecewise derivative — undefined at x=3 (kink)
    dg   = 1.5*x**2 - 3.0*x          # g'(x)
    sign = np.where(g(x) >= 0, 1.0, -1.0)
    return sign * dg + 0.5

def f_gap(x):
    return f(x) - f(X_STAR)

In [ ]:
xs_plot = np.linspace(-1, 4.5, 600)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- f(x) ---
ax1.plot(xs_plot, f(xs_plot), linewidth=2, label='f(x)')
ax1.axvline(3.0,    linestyle='--', color='grey',      linewidth=1.2, label='x = 3  (kink)')
ax1.axvline(X_STAR, linestyle='--', color='steelblue', linewidth=1.2,
            label=f'x* \u2248 {X_STAR:.4f}  (global min)')
ax1.scatter([X_STAR], [f(X_STAR)], color='steelblue', zorder=5, s=60)
ax1.set_title('Objective  f(x) = |g(x)| + 0.5x')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.legend(fontsize=9)

# --- f'(x) ---
ax2.plot(xs_plot, df(xs_plot), linewidth=2, color='darkorange')
ax2.axvline(3.0, linestyle='--', color='grey', linewidth=1.2, label='x = 3  (kink)')
ax2.axhline(0,   color='black', linewidth=0.8)
ax2.set_title("Derivative  f'(x)  (piecewise)")
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_01_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations from the landscape.**

The global minimizer is $x^\star = 1 - \tfrac{2\sqrt{3}}{3} \approx -0.1547$, where $f(x^\star) \approx -0.0396$.

**Why $x = 3$ is non-differentiable.**  
The one-sided limits of $f'$ at $x = 3$ differ:

$$\lim_{x \to 3^-} f'(x) = -g'(3) + \tfrac{1}{2} = -4.5 + 0.5 = -4,$$
$$\lim_{x \to 3^+} f'(x) = +g'(3) + \tfrac{1}{2} = +4.5 + 0.5 = +5.$$

**What the subdifferential $\partial f(3) = [-4,\, 5]$ means for GD.**  
Any scalar $d \in [-4, 5]$ is a valid subgradient at $x = 3$. A subgradient method would pick one element of this interval and take the step $x_{t+1} = x_t - \eta\, d$. Plain GD is undefined at the kink; in practice, GD reaches $x = 3$ with measure zero and the plot uses the right-sided value $d = 5$ by the `g(x) >= 0` convention.

## 4. Gradient Check

**What a finite-difference check is.**  
A finite-difference (FD) check approximates the derivative numerically:

$$f'(x_0) \approx \frac{f(x_0 + \varepsilon) - f(x_0 - \varepsilon)}{2\varepsilon}, \qquad \varepsilon = 10^{-6}.$$

The centered formula has $O(\varepsilon^2)$ error for smooth functions. If the analytic `df` is correct, the absolute error between the two estimates should be $\ll 10^{-4}$ on smooth branches.

**Why it matters.**  
A failing gradient check is the first sign of a coding error in `df`. Passing the check (on smooth test points) gives high confidence that `df` is correct before we rely on it inside GD.

**Test points** are chosen to cover both smooth branches and the global minimizer, while **excluding** $x = 3$ (the kink), where FD is not meaningful because the function is non-differentiable.

In [ ]:
test_points = [-0.8, -0.1547, 0.5, 1.0, 2.5, 3.5, 4.2]

print('=== Finite-difference gradient check ===')
print(f"{'x':>8}  {'analytic':>12}  {'finite-diff':>12}  {'abs error':>10}")
print('-' * 50)
for x0 in test_points:
    fd  = (f(x0 + EPS) - f(x0 - EPS)) / (2 * EPS)
    ana = float(df(x0))
    print(f"{x0:>8.4f}  {ana:>12.6f}  {fd:>12.6f}  {abs(ana-fd):>10.2e}")

**Results.**  
All absolute errors are well below $10^{-4}$, confirming that `df` is correctly implemented on both smooth branches ($g(x) < 0$ and $g(x) > 0$). The test point $x = -0.1547 \approx x^\star$ verifies the gradient is zero at the minimizer. The point $x = 3$ is excluded because finite differences are not meaningful at a kink — the centered formula would straddle the discontinuity and return a value close to $(5 + (-4))/2 = 0.5$, which corresponds to no real derivative.

## 5. Gradient Descent — Implementation

The GD update rule is

$$x_{t+1} = x_t - \eta\, f'(x_t).$$

We include two early-exit conditions:

1. **Convergence**: $f(x_t) - f(x^\star) < \text{TOL}$ — the objective gap is within tolerance.
2. **Divergence guard**: $|x_t| > 10^4$ — the iterate has left a recoverable region; we record `NaN` and stop.

In [ ]:
def gd(x0, eta, T=T_MAX):
    xs = [x0]
    x  = x0
    for _ in range(T):
        x = x - eta * df(x)
        xs.append(x)
        if f_gap(x) < TOL:
            break
        if abs(x) > 1e4:        # divergence guard
            xs.append(np.nan)
            break
    return np.array(xs)

In [ ]:
x0s    = [-0.8, 0.0, 0.5, 1.5, 2.8, 3.5]
T_plot = 60
eta_traj = 0.05

fig, ax = plt.subplots(figsize=(9, 5))

for x0 in x0s:
    xs = gd(x0, eta_traj, T=T_plot)
    ax.plot(xs[:T_plot], label=f'x0 = {x0}')

ax.axhline(X_STAR, linestyle='--', color='black', linewidth=1.2,
           label=f'x* \u2248 {X_STAR:.4f}')
ax.set_title('GD Trajectories — Multiple Initializations  (\u03b7 = 0.05, first 60 steps)')
ax.set_xlabel('Iteration')
ax.set_ylabel('x_t')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_02_gd_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

**Basin structure and trajectory behavior.**

- **$x_0 \in \{-0.8,\, 0.0,\, 0.5,\, 1.5\}$**: All four converge smoothly to $x^\star \approx -0.155$. These points lie below the basin boundary and the gradient consistently points toward the minimizer.

- **$x_0 = 2.8$ and $x_0 = 3.5$**: Neither converges within 5000 steps. Both oscillate persistently near the kink $x = 3$, bouncing between the $g < 0$ and $g > 0$ branches without progress toward $x^\star$.

**Why the oscillation is permanent.**  
On the $g < 0$ branch, $f'(x) = -\tfrac{3}{2}x^2 + 3x + \tfrac{1}{2}$ has two zeros: the global minimizer $x^\star \approx -0.155$ and a *local maximum* of $f$ at

$$x_{\text{max}} = 1 + \tfrac{2\sqrt{3}}{3} \approx 2.155.$$

For $x \in (2.155,\, 3)$, $f'(x) < 0$, so GD moves **right** toward $x = 3$. For $x > 3$ (the $g > 0$ branch), $f'(x) > 0$ always, so GD moves **left** back toward $x = 3$. The kink acts as a **stable trap** for any initialization above the basin boundary.

**Basin boundary summary.**

$$x_0 < x_{\text{max}} \approx 2.155 \implies \text{GD} \to x^\star, \qquad x_0 > x_{\text{max}} \implies \text{GD oscillates near } x = 3.$$

In [ ]:
etas_sweep = [0.01, 0.05, 0.10, 0.20, 0.50]
x0_sweep   = 2.0
T_sweep    = 200

trajectories = {eta: gd(x0_sweep, eta, T=T_sweep) for eta in etas_sweep}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Top: x_t vs iteration
for eta in etas_sweep:
    xs = trajectories[eta]
    ax1.plot(xs[:T_sweep], label=f'\u03b7 = {eta}')
ax1.axhline(X_STAR, linestyle='--', color='black', linewidth=1,
            label=f'x* \u2248 {X_STAR:.4f}')
ax1.set_title('GD Step-Size Sweep — x_t vs Iteration')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('x_t')
ax1.legend(fontsize=9)

# Bottom: f_gap on log scale
for eta in etas_sweep:
    xs = trajectories[eta]
    gaps = np.maximum(f_gap(np.array(xs[:T_sweep])), 1e-12)
    ax2.plot(gaps, label=f'\u03b7 = {eta}')
ax2.axhline(TOL, linestyle='--', color='black', linewidth=1, label=f'TOL = {TOL}')
ax2.set_yscale('log')
ax2.set_title('GD Step-Size Sweep — f_gap(x_t)  (log scale)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('f_gap(x_t)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_03_step_size_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

**Step-size sensitivity and the Lipschitz bound.**

For a function with $L$-Lipschitz gradient, GD with $\eta < 2/L$ is guaranteed to decrease $f$ at every step. For our piecewise $f$:

- Near the minimizer $x^\star$: $|f''(x^\star)| = |-3x^\star + 3| \approx 3.46$, giving $\eta < 2/3.46 \approx 0.58$.
- Near the kink at $x = 3$: $|f''| \to 6$ from both sides, giving the tighter bound $\eta < 2/6 \approx 0.33$.

The sweep starts at $x_0 = 2.0$, which is **below the basin boundary** ($2.0 < 2.155$), so all five step sizes converge to $x^\star$.  Speed varies as expected:

- **$\eta = 0.01$**: Converges but slowly; the gap is still above TOL after 200 steps.
- **$\eta = 0.05$–$0.10$**: Efficient convergence; the log-gap plot shows steady linear (on log scale) decrease.
- **$\eta = 0.20$**: Faster convergence, still within the local stability bound.
- **$\eta = 0.50$**: Exceeds the kink-region bound ($0.50 > 0.33$) but converges from $x_0 = 2.0$ because the trajectory stays away from $x = 3$. An initialization in $(2.155, 3)$ with $\eta = 0.50$ would produce larger oscillations near the kink.

## 8. Quadratic Baseline — Stability Reference

The convex quadratic $q(x) = \tfrac{1}{2}x^2$ with $q'(x) = x$ has a global Lipschitz constant $L = 1$, giving the clean stability condition $\eta < 2/L = 2$. This is the simplest benchmark for understanding step-size behavior before moving to non-smooth objectives.

In [ ]:
def q(x):
    return 0.5 * x**2

def dq(x):
    return x

def gd_quadratic(x0, eta, T=T_MAX):
    xs = [x0]
    x  = x0
    for _ in range(T):
        x = x - eta * dq(x)
        if abs(q(x) - q(0.0)) < TOL:
            xs.append(x)   # record the converged point before stopping
            break
        if abs(x) > 1e6:
            xs.append(np.nan)
            break
        xs.append(x)
    return np.array(xs)

x0_quad      = 4.0
stable_etas  = [0.05, 0.10, 0.15, 0.20]
unstable_eta = 2.1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Top: stable step sizes
for eta in stable_etas:
    xs = gd_quadratic(x0_quad, eta)
    ax1.plot(xs, label=f'\u03b7 = {eta}')
ax1.set_title('Quadratic GD — Stable Step Sizes  (0 < \u03b7 < 2)')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('x_t')
ax1.legend()

# Bottom: divergent step size
xs_div = gd_quadratic(x0_quad, unstable_eta)
ax2.plot(xs_div, color='purple', label=f'\u03b7 = {unstable_eta}')
ax2.set_title('Quadratic GD — Divergence  (\u03b7 = 2.1 > 2)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('x_t')
ax2.legend()

plt.tight_layout()
plt.savefig('fig_04_quadratic_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

For $q(x) = \tfrac{1}{2}x^2$, the GD update is $x_{t+1} = (1-\eta)x_t$, which converges if and only if $|1-\eta| < 1$, i.e., $0 < \eta < 2$. Step sizes $\eta \in \{0.05, 0.10, 0.15, 0.20\}$ all satisfy this and converge to $x^\star = 0$ (faster as $\eta$ grows). At $\eta = 2.1$, the iterate follows $x_t = (-1.1)^t x_0$, which diverges geometrically — exactly as shown.

## 9. Final Commentary

### 9.1 What the Piecewise Structure Taught Us

Working with $f(x) = |g(x)| + \tfrac{1}{2}x$ exposed three ideas that smooth objectives hide:

- **Kink and subdifferential.** At $x = 3$, $\partial f(3) = [-4, 5]$ replaces the scalar gradient. Plain GD is undefined at the kink; in floating-point practice the iterate never lands exactly there, so the right-sided value $d = 5$ is used by the `g(x) >= 0` convention.

- **Finite basin of attraction.** The global minimizer $x^\star \approx -0.155$ attracts only initializations with $x_0 < x_{\text{max}} \approx 2.155$. The point $x_{\text{max}} = 1 + \tfrac{2\sqrt{3}}{3}$ is a local *maximum* of $f$ on the $g < 0$ branch — an unstable fixed point that acts as the basin boundary. Initializations above it are permanently trapped in kink-crossing oscillation near $x = 3$.

- **Branch-aware derivative.** The sign of $g(x)$ must be tracked explicitly. Any implementation that ignores the piecewise branching produces wrong gradients on the $g > 0$ branch, causing GD to diverge or converge to the wrong point.

### 9.2 Step-Size Sensitivity Summary

| Setting | Local $L$ | Stability bound $\eta < 2/L$ |
|---------|-----------|------------------------------|
| Quadratic $q$, global | $1$ | $\eta < 2.0$ |
| Piecewise $f$, near $x = 3$ | $\approx 6$ | $\eta < 0.33$ |
| Piecewise $f$, near $x^\star$ | $\approx 3.46$ | $\eta < 0.58$ |

The quadratic gives a single global criterion. For the piecewise objective, the bound is *local*: a step size safe near the minimizer ($\eta < 0.58$) can still cause oscillations if the trajectory passes near the kink ($\eta < 0.33$ required there). The step-size sweep from $x_0 = 2.0$ avoids the kink, so all five $\eta$ values converge — but even $\eta = 0.50$ would oscillate from an initialization in $(2.155, 3)$.

### 9.3 Takeaways and Open Questions

**Three takeaways:**

1. A finite-difference gradient check is inexpensive and should be the first test for any new derivative implementation. All errors here were below $10^{-8}$ on smooth branches — the analytic `df` is correct.
2. Non-smooth objectives can have **attracting traps** that are not global optima. The kink at $x = 3$ is not a critical point of $f$, yet GD gets stuck there permanently for $x_0 > 2.155$. This cannot happen for smooth strongly-convex functions.
3. The quadratic baseline is not just a warm-up — the stability criterion $\eta < 2/L$ derived there is the same principle that governs every smooth region of the piecewise objective, and it tightens near the kink.

**Two open questions:**

1. Would a *projected subgradient method* — which explicitly selects the minimum-norm element of $\partial f(x)$ at each step — escape the kink trap from $x_0 = 2.8$, or does the trap persist for any fixed step size?
2. Could adding a small amount of noise (as in SGD) allow the iterate to cross the basin boundary $x_{\text{max}} \approx 2.155$ from above and escape the kink oscillation, effectively using stochasticity as an exploration mechanism?